<span style="font-family: Arial; font-weight:bold;font-size:2.1em;color:#fff;">-----------------------------------
    
<span style="font-family: Arial; font-weight:bold;font-size:2.1em;color:#fa9200;">Use Indicators data to build LSTM Equivalent
    
<span style="font-family: Arial; font-weight:bold;font-size:2.1em;color:#fff;">-----------------------------------

In [3]:
#!pip install torch

### Import Packages

In [14]:
import torch
import itertools
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from datetime import date, datetime, timedelta
import os
import re
import matplotlib as mpl
import matplotlib.pyplot as plt
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
import statsmodels.formula.api as sm
import statsmodels.stats.power as smp
import yfinance as yf
import plotly.express as px
import plotly.graph_objects as go
import plotly
from scipy.signal import argrelextrema
from collections import defaultdict
import sqlite3
import warnings
warnings.filterwarnings("ignore")
import Indicators
import Measurement
import Charts

### Add Variables

In [3]:
M = 80
K = 500
window = 7
smoothing = 7
events = {'ihs_event':'bull','hs_event':'bear','fw_event':'bull','rw_event':'bear'}
bound = 0.03
SMAs = [30,60,90]

### Run Functions to Get Data and Indicators

In [4]:
df3, final4 = Measurement.main(events, SMAs, smoothing, window, M, K, bound)

BRK.B: No data found, symbol may be delisted
'DataFrame' object has no attribute 'Datetime'
BF.B: Period '500d' is invalid, must be one of ['1mo', '3mo', '6mo', 'ytd', '1y', '2y', '5y', '10y', 'max']
'DataFrame' object has no attribute 'Datetime'


### Keep only columns Needed

In [ ]:
cols = ['ticker','date','open','high','low','close','SMA30','SMA60','SMA90','rw_event','rw_event_none','rw_event_group',
        'rw_event_start_time','rw_event_end_time']
df4 = df3[cols].copy()
df4.head(2)

### Test Results for One Inidicator

In [6]:
event = 'rw_event'
final4.head(2)

,index,ticker,event,event_observations,event_start_time,event_end_time,after_event_observations,after_event_mean,after_event_start_time,after_event_end_time,count,mean,min,median,max,Indicator,event_count,event_success,stock_success
0,0,A,rw_event,36.0,2023-10-31 14:30:00,2023-11-07 14:30:00,1.0,-0.049453,2023-11-07 14:30:00,2023-11-24 09:30:00,2752.0,-0.045873,-0.223545,-0.035407,0.056347,1,1.0,0.762963,1.0
1,1,A,rw_event,51.0,2023-12-11 10:30:00,2023-12-20 11:30:00,1.0,-0.088146,2023-12-20 11:30:00,2024-01-08 13:30:00,2752.0,-0.045873,-0.223545,-0.035407,0.056347,1,1.0,0.762963,1.0


### Join with Final to identify Successful Indicator Results

In [7]:
rw_final4 = final4[(final4.event == event)&(final4.Indicator==1)].copy()
test = pd.merge(df4, rw_final4[['event_start_time','ticker','Indicator']], left_on = [event + '_start_time','ticker'], right_on = ['event_start_time','ticker'], 
    how='left').rename(columns={'Indicator': event + '_indicator'}).drop(['event_start_time'], axis=1)
test[event+'_indicator'] = test[event + '_indicator'].fillna(0)
test[event+'_indicator2'] = np.where((test[event + '_indicator'] == 1) & (test[event + '_indicator'].shift(-1) == 0), 1, 0)
test.head(2)

,ticker,date,open,high,low,close,SMA30,SMA60,SMA90,rw_event,rw_event_none,rw_event_group,rw_event_start_time,rw_event_end_time,rw_event_indicator,rw_event_indicator2
0,ZTS,2022-09-06 09:30:00,157.759995,158.720001,156.649994,157.949997,NaN,NaN,NaN,0,1,1,2022-09-06 09:30:00,2024-08-30 15:30:00,0.0,0
1,ZTS,2022-09-06 10:30:00,157.839996,159.524994,157.669998,158.429993,NaN,NaN,NaN,0,1,1,2022-09-06 09:30:00,2024-08-30 15:30:00,0.0,0


### Validate the Join

In [8]:
test.groupby(['rw_event'])['rw_event_indicator'].describe()

,count,mean,std,min,25%,50%,75%,max
rw_event,,,,,,,,
0,1683662.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0
1,42821.0,0.372761,0.483545,0.0,0.0,0.0,1.0,1.0


### <ins>Tangent:</ins> Looking for Large Changes in Stock Price

In [9]:
smaList = [x for x in df4.columns if 'SMA' in x]
for s in smaList:
    test[s + '_chng'] = (test['close'] - test[s])/test['close']
test.head(2)

,ticker,date,open,high,low,close,SMA30,SMA60,SMA90,rw_event,rw_event_none,rw_event_group,rw_event_start_time,rw_event_end_time,rw_event_indicator,rw_event_indicator2,SMA30_chng,SMA60_chng,SMA90_chng
0,ZTS,2022-09-06 09:30:00,157.759995,158.720001,156.649994,157.949997,NaN,NaN,NaN,0,1,1,2022-09-06 09:30:00,2024-08-30 15:30:00,0.0,0,NaN,NaN,NaN
1,ZTS,2022-09-06 10:30:00,157.839996,159.524994,157.669998,158.429993,NaN,NaN,NaN,0,1,1,2022-09-06 09:30:00,2024-08-30 15:30:00,0.0,0,NaN,NaN,NaN


In [10]:
pDict = {'open':4,'high':2,'low':3,'close':1}
for x in pDict.keys():
    for y in pDict.keys():
        if pDict[x] < pDict[y]:
            test[x + '_' + y + '_chng'] = (test[x] - test[y])/test[x]
        else:
            continue

test.head(2)

,ticker,date,open,high,low,close,SMA30,SMA60,SMA90,rw_event,...,rw_event_indicator2,SMA30_chng,SMA60_chng,SMA90_chng,high_open_chng,high_low_chng,low_open_chng,close_open_chng,close_high_chng,close_low_chng
0,ZTS,2022-09-06 09:30:00,157.759995,158.720001,156.649994,157.949997,NaN,NaN,NaN,0,...,0,NaN,NaN,NaN,0.006048,0.013042,-0.007086,0.001203,-0.004875,0.008230
1,ZTS,2022-09-06 10:30:00,157.839996,159.524994,157.669998,158.429993,NaN,NaN,NaN,0,...,0,NaN,NaN,NaN,0.010563,0.011628,-0.001078,0.003724,-0.006912,0.004797


### Run PCA on specified event

In [11]:
test.dropna(inplace=True)
chngList = [x for x in test.columns if 'chng' in x]
X = test[chngList].values
X = StandardScaler().fit_transform(X)
pca_fit = PCA(n_components=2)
pcomp = pca_fit.fit_transform(X)
pcomp

array([[ 0.62747954,  0.62236198],
       [ 2.74162406, -1.06748257],
       [ 1.03990662,  1.29721881],
       ...,
       [-1.6912426 , -0.18377042],
       [-1.65764457, -0.52205894],
       [-1.72214327,  0.7684128 ]])

### ADD PCA to DF

In [12]:
test['pca1'] = pcomp[:,0]
test['pca2'] = pcomp[:,0]
test.head(2)

,ticker,date,open,high,low,close,SMA30,SMA60,SMA90,rw_event,...,SMA60_chng,SMA90_chng,high_open_chng,high_low_chng,low_open_chng,close_open_chng,close_high_chng,close_low_chng,pca1,pca2
106,ZTS,2022-09-22 14:30:00,149.380005,150.559998,149.309998,150.289993,153.710206,157.40608,158.338830,0,...,-0.047349,-0.053555,0.007837,0.008302,-0.000469,0.006055,-0.001797,0.006521,0.627480,0.627480
107,ZTS,2022-09-22 15:30:00,150.339996,150.514008,149.399994,149.460007,153.465206,157.12608,158.244497,0,...,-0.051292,-0.058775,0.001156,0.007401,-0.006292,-0.005888,-0.007052,0.000402,2.741624,2.741624


### ScatterPlot PCA Results

In [13]:
test2 = test.sample(frac=.1).copy()
fig = go.Figure()
fig.add_trace(go.Scatter(x=test2[test2[event]==0].pca1, 
                    y=test2[test2[event]==0].pca2,
                    mode='markers',
                    name='markers',
                    ))
fig.add_trace(go.Scatter(x=test2[test2[event]==1].pca1, 
                    y=test2[test2[event]==1].pca2,
                    mode='markers',
                    name='markers',
                    ))
plotly.offline.plot(fig, filename='C:/Users/rschaub/Downloads/test.html')

'C:/Users/rschaub/Downloads/test.html'

### Histogram of PCA Results

In [92]:
fig = px.histogram(test2, x="pca1", color=event, nbins=200)
plotly.offline.plot(fig, filename='C:/Users/rschaub/Downloads/test.html')

'C:/Users/rschaub/Downloads/test.html'

### Note: The Scatter plot appeared to show that the successful indicators have a tighter distribution, but upon further investigation the historgram appears to show that the distributions are in fact similar but there are less outliers in the successful data set, perhaps due to the smaller number of observations. We can confirm this by picking a significant sample size and using A/B testing

In [17]:
effect_size = 0.5  # The desired effect size (e.g., Cohen's d)
alpha = 0.05  # The significance level (e.g., 0.05)
power = 0.95  # The desired statistical power (e.g., 0.8)

# For a paired t-test
power_analysis = smp.TTestPower()
sample_size = power_analysis.solve_power(effect_size=effect_size, alpha=alpha, power=power)

# Round up to the nearest integer
sample_size = round(sample_size)

print("Required Sample Size:", sample_size)

Required Sample Size: 54
